In [2]:
import os
import json

import pandas as pd
import numpy as np

In [3]:
# Load data 

data_dir_path = "../data/realAWSCloudwatch"
labels_path = "../data/labels/combined_labels.json"

dfs = {}

# Load csv files into dataframes
for filename in os.listdir(data_dir_path):
    if filename.startswith("ec2_cpu_utilization"):
        df = pd.read_csv(os.path.join(data_dir_path, filename))
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        dfs[filename] = df

# Load labels and add anomaly_label column to dataframes
with open(labels_path, "r") as f:
    labels = json.load(f)

for filename, df in dfs.items():
    key = f"realAWSCloudwatch/{filename}"
    anomaly_timestamps = pd.to_datetime(labels.get(key, []), errors="coerce")
    df["anomaly_label"] = df["timestamp"].isin(anomaly_timestamps).astype("bool")

In [7]:
# Convert dataframes to datasets of sliding windows

# Arbitrary window and horizon sizes, can be tuned as a hyperparameter
W = 30 # 150 minutes
H = 5  # 25 minutes

def create_sliding_windows(df, W, H):
    X = []
    y = []
    for i in range(len(df) - W - H + 1):
        window = df["value"].iloc[i:i+W].values
        label = df["anomaly_label"].iloc[i+W:i+W+H].any()
        X.append(window)
        y.append(label)
    return np.array(X), np.array(y)

datasets = {}

for filename in dfs:
    df = dfs[filename]
    X, y = create_sliding_windows(df, W, H)
    datasets[filename] = (X, y)

In [ ]:
# Designing features 

